# Clase 129 — Pooling

El **pooling** reduce las dimensiones espaciales **sin parámetros
entrenables**. `MaxPooling2D` toma el máximo de cada ventana (preserva la
característica más fuerte y añade *invariancia local a translación*),
`AveragePooling2D` promedia, y `GlobalAveragePooling2D` colapsa cada feature
map a un solo valor `(batch, channels)` — el estándar moderno antes de la
cabeza `Dense`, en reemplazo de `Flatten`.

Requiere: `numpy`, `tensorflow` / `keras` (≥ 3.0). No se ejecuta aquí; código
idiomático y correcto por API.

## 1. `MaxPooling2D`: efecto en las dimensiones

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
keras.utils.set_random_seed(42)

entrada = keras.Input(shape=(28, 28, 32))
y = layers.MaxPooling2D(pool_size=2)(entrada)   # strides=2 por defecto
print("MaxPooling2D(2):", entrada.shape, "->", y.shape)   # (None, 14, 14, 32)

## 2. Max-pool vs average-pool sobre un tensor concreto

In [ ]:
x = np.array([[1.,  2.,  5.,  6.],
              [3.,  4.,  7.,  8.],
              [9.,  10., 13., 14.],
              [11., 12., 15., 16.]], dtype="float32").reshape(1, 4, 4, 1)

mx = layers.MaxPooling2D(2)(x).numpy().reshape(2, 2)
av = layers.AveragePooling2D(2)(x).numpy().reshape(2, 2)
print("max-pool 2x2 (el máximo de cada bloque):")
print(mx)
print("avg-pool 2x2 (el promedio de cada bloque):")
print(av)

## 3. `GlobalAveragePooling2D` reemplaza a `Flatten`

`GlobalAveragePooling2D` colapsa cada feature map a un escalar, evitando el
`Flatten` + `Dense` gigante y reduciendo drásticamente los parámetros.

In [ ]:
flat = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
gap = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation="softmax"),
])
print("Flatten + Dense(128):", flat.count_params())
print("GlobalAveragePooling:", gap.count_params())   # muchos menos parámetros

## 4. Invariancia local a translación

Si la característica se desplaza 1 píxel dentro de la misma ventana, el máximo
no cambia: esa es la invariancia local que aporta el max-pool.

In [ ]:
base = np.zeros((1, 4, 4, 1), dtype="float32");  base[0, 0, 0, 0] = 9.0
shift = np.zeros((1, 4, 4, 1), dtype="float32"); shift[0, 0, 1, 0] = 9.0

m_base = layers.MaxPooling2D(2)(base).numpy().reshape(2, 2)[0, 0]
m_shift = layers.MaxPooling2D(2)(shift).numpy().reshape(2, 2)[0, 0]
print("max-pool base :", m_base)    # 9.0
print("max-pool shift:", m_shift)   # 9.0 -> invariante al desplazamiento local

## 5. Pooling vs stride (approach moderno)

`Conv(stride=1) + Pool(2)` y `Conv(stride=2)` producen el mismo downsampling.
ResNet/ConvNeXt prefieren el stride en la conv y eliminan el pool intermedio.

In [ ]:
entrada = keras.Input(shape=(28, 28, 1))
pool = layers.MaxPooling2D(2)(
    layers.Conv2D(32, 3, strides=1, padding="same")(entrada))
strd = layers.Conv2D(32, 3, strides=2, padding="same")(entrada)
print("Conv(s=1) + Pool(2):", pool.shape)   # (None, 14, 14, 32)
print("Conv(s=2)          :", strd.shape)   # (None, 14, 14, 32)

## 6. `GlobalAveragePooling2D` y padding con H impar

Con H impar, `padding='valid'` recorta y `padding='same'` agrega un
row/col de zeros.

In [ ]:
entrada = keras.Input(shape=(7, 7, 64))
g = layers.GlobalAveragePooling2D()(entrada)
print("GlobalAveragePooling2D:", entrada.shape, "->", g.shape)   # (None, 64)

odd = keras.Input(shape=(7, 7, 8))
print("valid:", layers.MaxPooling2D(2, padding="valid")(odd).shape)  # (None,3,3,8)
print("same :", layers.MaxPooling2D(2, padding="same")(odd).shape)   # (None,4,4,8)

## Ejercicios

1. **Shape de MaxPool**: aplicá `MaxPooling2D(2)` a un tensor `(1, 28, 28, 32)`
   y verificá la salida `(1, 14, 14, 32)`.
2. **Max vs Avg**: sobre el tensor 4×4 del notebook, comprobá que max-pool toma
   el máximo y avg-pool el promedio de cada bloque.
3. **GAP vs Flatten**: compará los parámetros de la arquitectura con
   `Flatten + Dense(128)` frente a la de `GlobalAveragePooling2D`.
4. **Pool vs stride**: verificá que `Conv(s=1)+Pool(2)` y `Conv(s=2)` dan el
   mismo shape de salida.

## Conclusiones

- El pooling reduce las dimensiones espaciales y **no tiene parámetros entrenables**.
- Max-pool preserva la activación más fuerte y aporta invariancia local a translación.
- `GlobalAveragePooling2D` reemplaza a `Flatten` y reduce muchísimo los parámetros.
- Pool(2) y stride=2 producen el mismo downsampling; el enfoque moderno usa stride.
- Con H impar, `'valid'` recorta y `'same'` agrega zeros.